# Phase 9: Model evaluation and comparison

Compare out-of-fold RUL predictions from engine-grouped cross-validation. Metrics include MAE, RMSE, R², and the asymmetric NASA C-MAPSS score; plots are saved under `reports/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.evaluation.comparison import compare_model_results, plot_prediction_diagnostics
from src.features.feature_engineering import build_feature_matrix
from src.features.target_engineering import prepare_fd_dataset
from src.models.baseline import run_baseline_model_suite

raw_dir = Path('../data/raw/CMAPSSData')
fd_data = prepare_fd_dataset('FD001', raw_dir=raw_dir)
feature_matrix = build_feature_matrix(fd_data['train'], target_column='rul_uncapped', windows=(3, 5, 10))
results = run_baseline_model_suite(
    feature_matrix,
    target_column='rul_uncapped',
    group_column='engine_id',
    n_splits=3,
    include_models=['linear_regression', 'ridge', 'random_forest', 'svr'],
)

comparison = compare_model_results(results, rank_by='rmse')
Path('../reports').mkdir(parents=True, exist_ok=True)
comparison.to_csv('../reports/phase9_model_comparison.csv', index=False)
comparison

In [ ]:
best_model_name = comparison.iloc[0]['model']
best_result = results[best_model_name]
plot_prediction_diagnostics(
    best_result.true_values,
    best_result.predictions,
    '../reports/phase9_best_model_diagnostics.png',
    title=f'{best_model_name}: engine-grouped out-of-fold predictions',
)
print(f'Best model by pooled RMSE: {best_model_name}')